# Your First Free Infinite Loop

A self-referencing feedback loop that generates Q&A data on any topic using **free OpenRouter models**.  
**Zero cost · Runs anywhere · Produces a fine-tuning dataset.**

---

**What this notebook does:**
1. Asks a free LLM a question about your chosen topic
2. Stores the answer
3. Uses that answer to generate a *better* next question
4. Repeats indefinitely — each iteration informed by the last
5. Exports the accumulated data as a fine-tuning dataset

**From:** [pocoo.vaked.dev — Your first free infinite loop](https://pocoo.vaked.dev/posts/2026-06-24-your-first-free-infinite-loop.html)  
**Theory:** [The genesis contract, formally](https://pocoo.vaked.dev/posts/2026-06-24-genesis-contract-formally.html)  
**Source:** [github.com/peterlodri-sec/ultrawhale/examples](https://github.com/peterlodri-sec/ultrawhale/tree/main/examples)

---

> **Genesis contract for this loop**  
> *Reducing:* gaps in understanding of a topic  
> *Valid iteration:* answer introduces ≥1 concept not in previous 5 answers  
> *Terminate:* you can explain the topic to a non-expert in 5 minutes  
> *Invariants:* answers must be checkable; if the loop cannot be corrected by evidence in 3 turns, stop and inject

In [ ]:
# ── Cell 1: Install & imports ────────────────────────────────────────────────
from __future__ import annotations  # enables str | None on Python 3.9

!pip install requests matplotlib -q

import os, json, time, requests
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

print('✓ Ready — Python', __import__('sys').version.split()[0])

In [ ]:
# ── Cell 2: API key ──────────────────────────────────────────────────────────
# Get a FREE key at https://openrouter.ai → Sign up → API Keys → Create
# No credit card required. Free models never charge.
#
# In Google Colab: click the 🔑 (Secrets) icon in the left sidebar
#                  Add secret name=OPENROUTER_KEY, value=sk-or-v1-...
# Locally:         export OPENROUTER_KEY=sk-or-v1-...

OPENROUTER_KEY = os.environ.get('OPENROUTER_KEY', '')

if not OPENROUTER_KEY:
    try:
        from google.colab import userdata
        OPENROUTER_KEY = userdata.get('OPENROUTER_KEY') or ''
    except Exception:
        pass

if not OPENROUTER_KEY:
    OPENROUTER_KEY = input('Paste your OpenRouter API key (sk-or-v1-...): ').strip()

if OPENROUTER_KEY:
    print(f'✓ Key set: {OPENROUTER_KEY[:12]}...')
else:
    print('⚠  No key set. Get one free at https://openrouter.ai')

In [ ]:
# ── Cell 3: Verify which free models are available ───────────────────────────
# Checks the two models used by this notebook.
# If one fails, the loop will automatically use the other.

CANDIDATE_MODELS = [
    'openai/gpt-oss-20b:free',
    'liquid/lfm-2.5-1.2b-instruct:free',
    'meta-llama/llama-3.2-3b-instruct:free',
]

def _probe(model: str) -> bool:
    try:
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': f'Bearer {OPENROUTER_KEY}', 'Content-Type': 'application/json'},
            json={'model': model, 'messages': [{'role': 'user', 'content': 'Ping. Reply: pong'}], 'max_tokens': 10},
            timeout=20,
        )
        choices = r.json().get('choices')
        return bool(choices and choices[0].get('message', {}).get('content'))
    except Exception:
        return False

FREE_MODELS = []
for m in CANDIDATE_MODELS:
    ok = _probe(m)
    status = '✓' if ok else '✗'
    print(f'{status} {m}')
    if ok:
        FREE_MODELS.append(m)

print()
if FREE_MODELS:
    print(f'Using {len(FREE_MODELS)} working model(s): {[m.split("/")[1] for m in FREE_MODELS]}')
else:
    print('⚠  No working models found. Check your API key or visit https://openrouter.ai/models?max_price=0')

In [ ]:
# ── Cell 4: Configure your loop ─────────────────────────────────────────────
# Change TOPIC to whatever you want to learn about deeply.
# The loop will explore it iteratively, building on each answer.

TOPIC = 'techniques for compressing LLM context windows and tool outputs'
# Other ideas:
# TOPIC = 'how neural field equations produce visual hallucinations'
# TOPIC = 'Rust ownership model edge cases and common beginner mistakes'
# TOPIC = 'post-quantum cryptography primitives and their tradeoffs'
# TOPIC = 'how transformer attention patterns change at different layers'

OUTPUT_FILE     = 'loop_output.jsonl'  # results saved here after each iteration
MAX_ITERATIONS  = 50    # set to None to run forever (Ctrl+C to stop safely)
INTERVAL_SEC    = 8     # seconds between API calls (be kind to free tier)

print(f'Topic:     {TOPIC}')
print(f'Output:    {OUTPUT_FILE}')
print(f'Max iter:  {MAX_ITERATIONS or "∞"}')
print(f'Interval:  {INTERVAL_SEC}s')

In [ ]:
# ── Cell 5: Core loop functions ──────────────────────────────────────────────

def ask(prompt: str, model: str, max_tokens: int = 512) -> str | None:
    """Call one free OpenRouter model. Returns None on any failure."""
    try:
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {OPENROUTER_KEY}',
                'HTTP-Referer': 'https://pocoo.vaked.dev',
                'X-Title': 'free-infinite-loop-notebook',
                'Content-Type': 'application/json',
            },
            json={
                'model': model,
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': max_tokens,
                'temperature': 0.7,
            },
            timeout=30,
        )
        r.raise_for_status()
        choices = r.json().get('choices')
        if not choices:
            return None
        return choices[0]['message']['content'].strip()
    except requests.HTTPError as e:
        if e.response.status_code == 429:
            print(f'    [rate limit] {model} — sleeping 10s...')
            time.sleep(10)
        return None
    except Exception:
        return None


def generate_question(topic: str, knowledge: list[str], model: str) -> str | None:
    """Generate the next question, informed by what the loop already knows."""
    if not knowledge:
        # First iteration: ask for the single best entry point
        prompt = (
            f'Ask the most important first question about: {topic}\n'
            f'Target a specific mechanism, concept, or tradeoff — not a meta-question.\n'
            f'Just the question, nothing else.'
        )
    else:
        # Subsequent iterations: build on accumulated knowledge
        context = '\n'.join(f'  • {k[:150]}' for k in knowledge[-5:])
        prompt = (
            f'You are studying: {topic}\n\n'
            f'What you have learned so far:\n{context}\n\n'
            f'What specific question would most deepen your understanding from here?\n'
            f'The question should go further than what you already know.\n'
            f'Just the question, nothing else.'
        )
    return ask(prompt, model, max_tokens=128)


def answer_question(question: str, model: str) -> str | None:
    """Get a thorough answer to the question."""
    prompt = (
        f'Give a clear, accurate, and thorough answer to this question.\n'
        f'If you are uncertain about anything, say so explicitly.\n\n'
        f'Question: {question}'
    )
    return ask(prompt, model, max_tokens=600)


print('✓ Functions defined: ask(), generate_question(), answer_question()')

In [ ]:
# ── Cell 6: Run the loop ─────────────────────────────────────────────────────
# Press the stop button (■) in Colab, or Ctrl+C locally, to stop.
# All completed iterations are already saved — no data is lost on interrupt.

if not FREE_MODELS:
    print('⚠  No working models — run Cell 3 first and check your API key.')
else:
    out           = Path(OUTPUT_FILE)
    knowledge: list[str] = []
    records: list[dict]  = []
    q_model_idx   = 0
    a_model_idx   = 1
    iteration     = 0

    # Resume from existing file if it exists
    if out.exists():
        existing = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]
        if existing:
            print(f'▶ Resuming from {len(existing)} existing records')
            records = existing
            # Rebuild knowledge from last 10 records
            knowledge = [
                f'Q: {r["question"][:100]}  A: {r["answer"][:150]}'
                for r in existing[-10:]
            ]
            iteration = existing[-1].get('iteration', len(existing) - 1) + 1

    print(f'Starting: {TOPIC!r}')
    print(f'Output  → {out.absolute()}')
    print()
    print(f'{"Iter":>5}  {"Model":<26}  Question preview')
    print('─' * 80)

    try:
        while MAX_ITERATIONS is None or iteration < MAX_ITERATIONS:
            # Alternate question/answer models for diversity
            q_model = FREE_MODELS[q_model_idx % len(FREE_MODELS)]
            a_model = FREE_MODELS[a_model_idx % len(FREE_MODELS)]

            # Generate the next question
            question = generate_question(TOPIC, knowledge, q_model)
            if not question:
                print(f'{iteration:>5}  [question generation failed — retrying]')
                time.sleep(INTERVAL_SEC)
                continue

            # Get the answer
            answer = answer_question(question, a_model)
            if not answer:
                print(f'{iteration:>5}  [answer generation failed — skipping]')
                q_model_idx += 1
                a_model_idx += 1
                time.sleep(INTERVAL_SEC)
                continue

            # Accumulate knowledge (for the next question)
            knowledge.append(f'Q: {question[:100]}  A: {answer[:200]}')

            # Build and save record
            record = {
                'id':        f'loop-{iteration:05d}',
                'question':  question,
                'answer':    answer,
                'q_model':   q_model,
                'a_model':   a_model,
                'topic':     TOPIC,
                'timestamp': datetime.now(timezone.utc).isoformat(),
                'iteration': iteration,
                'answer_words': len(answer.split()),
                'question_words': len(question.split()),
            }
            records.append(record)
            with out.open('a') as f:
                f.write(json.dumps(record) + '\n')

            # Print progress
            q_short  = question[:52] + '...' if len(question) > 55 else question
            m_short  = q_model.split('/')[1][:24]
            print(f'{iteration:>5}  {m_short:<26}  {q_short!r}')

            q_model_idx += 1
            a_model_idx += 1
            iteration   += 1
            time.sleep(INTERVAL_SEC)

    except KeyboardInterrupt:
        pass

    print(f'\n✓ Stopped at iteration {iteration}. {len(records)} total records → {out}')

In [ ]:
# ── Cell 7: Inspect results ──────────────────────────────────────────────────
# Run this any time — even while the loop is paused.

out = Path(OUTPUT_FILE)
if not out.exists():
    print('No output file yet — run Cell 6 first.')
else:
    all_records = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]
    print(f'Total records: {len(all_records)}')
    if all_records:
        avg_q = sum(r.get('question_words', len(r['question'].split())) for r in all_records) / len(all_records)
        avg_a = sum(r.get('answer_words', len(r['answer'].split())) for r in all_records) / len(all_records)
        print(f'Avg question length: {avg_q:.0f} words')
        print(f'Avg answer length:   {avg_a:.0f} words')
        print()
        print('── First 2 records ────────────────────────────────────────────────')
        for r in all_records[:2]:
            print(f'[{r["id"]}]')
            print(f'  Q: {r["question"]}')
            print(f'  A: {r["answer"][:200]}...' if len(r['answer']) > 200 else f'  A: {r["answer"]}')
            print()
        if len(all_records) > 2:
            print('── Most recent record ─────────────────────────────────────────────')
            r = all_records[-1]
            print(f'[{r["id"]}]')
            print(f'  Q: {r["question"]}')
            print(f'  A: {r["answer"][:200]}...' if len(r['answer']) > 200 else f'  A: {r["answer"]}')

In [ ]:
# ── Cell 8: Visualize loop evolution ────────────────────────────────────────
# Shows how question and answer complexity grows over iterations.
# A proxy for how much the loop is actually learning.

import matplotlib.pyplot as plt
import matplotlib

out = Path(OUTPUT_FILE)
if not out.exists() or not out.stat().st_size:
    print('No data yet — run Cell 6 first.')
else:
    all_records = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]

    if len(all_records) < 3:
        print(f'Only {len(all_records)} records — run more iterations for a meaningful chart.')
    else:
        matplotlib.rcParams.update({
            'figure.facecolor': '#070b16',
            'axes.facecolor':   '#0a0a14',
            'axes.edgecolor':   '#26304a',
            'text.color':       '#e0e8f5',
            'axes.labelcolor':  '#6878a0',
            'xtick.color':      '#6878a0',
            'ytick.color':      '#6878a0',
            'grid.color':       '#26304a',
            'grid.linestyle':   ':',
        })

        iters   = [r['iteration'] for r in all_records]
        q_words = [r.get('question_words', len(r['question'].split())) for r in all_records]
        a_words = [r.get('answer_words',   len(r['answer'].split()))   for r in all_records]

        # Rolling average (window=5) to smooth noise
        def rolling(vals, w=5):
            return [sum(vals[max(0,i-w):i+1]) / len(vals[max(0,i-w):i+1]) for i in range(len(vals))]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5), facecolor='#070b16')
        fig.suptitle(f'Loop evolution: {TOPIC[:65]}', color='#e0e8f5', fontsize=10)

        ax1.plot(iters, q_words, color='#26304a', linewidth=0.8, alpha=0.5)
        ax1.plot(iters, rolling(q_words), color='#00d4ff', linewidth=2, label='question length')
        ax1.set_ylabel('words', fontsize=9)
        ax1.legend(fontsize=8, facecolor='#14141f', edgecolor='#26304a', labelcolor='#e0e8f5')
        ax1.grid(True, axis='y')
        for sp in ['top','right']: ax1.spines[sp].set_visible(False)

        ax2.plot(iters, a_words, color='#26304a', linewidth=0.8, alpha=0.5)
        ax2.plot(iters, rolling(a_words), color='#00e660', linewidth=2, label='answer length')
        ax2.set_xlabel('iteration', fontsize=9)
        ax2.set_ylabel('words', fontsize=9)
        ax2.legend(fontsize=8, facecolor='#14141f', edgecolor='#26304a', labelcolor='#e0e8f5')
        ax2.grid(True, axis='y')
        for sp in ['top','right']: ax2.spines[sp].set_visible(False)

        plt.tight_layout()
        plt.savefig('loop_evolution.png', dpi=150, bbox_inches='tight', facecolor='#070b16')
        plt.show()
        print('✓ Chart saved → loop_evolution.png')

In [ ]:
# ── Cell 9: Export for fine-tuning ──────────────────────────────────────────
# Alpaca format is compatible with Unsloth, LM Studio, axolotl, llama-factory.
# Each loop record becomes one training example: instruction + output.

out = Path(OUTPUT_FILE)
if not out.exists():
    print('No data yet — run Cell 6 first.')
else:
    all_records = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]

    MIN_ANSWER_WORDS = 15  # filter out very short / failed answers

    alpaca = [
        {
            'instruction': r['question'],
            'input': '',
            'output': r['answer'],
        }
        for r in all_records
        if len(r.get('answer', '').split()) >= MIN_ANSWER_WORDS
    ]

    # Also export in ShareGPT format (preferred by some trainers)
    sharegpt = [
        {
            'conversations': [
                {'from': 'human', 'value': r['question']},
                {'from': 'gpt',   'value': r['answer']},
            ]
        }
        for r in all_records
        if len(r.get('answer', '').split()) >= MIN_ANSWER_WORDS
    ]

    base = OUTPUT_FILE.replace('.jsonl', '')
    alpaca_file   = f'{base}_alpaca.json'
    sharegpt_file = f'{base}_sharegpt.json'

    Path(alpaca_file).write_text(json.dumps(alpaca, indent=2))
    Path(sharegpt_file).write_text(json.dumps(sharegpt, indent=2))

    print(f'✓ Exported {len(alpaca)} records (filtered from {len(all_records)})')
    print(f'  Alpaca format   → {alpaca_file}')
    print(f'  ShareGPT format → {sharegpt_file}')
    print()
    print('Fine-tuning options:')
    print('  Unsloth (fast, free): https://github.com/unslothai/unsloth')
    print('  LM Studio (local GUI): https://lmstudio.ai')
    print('  Axolotl (flexible): https://github.com/axolotl-ai-cloud/axolotl')
    print('  Ollama (local inference after training): https://ollama.ai')

In [ ]:
# ── Cell 10: Analyze what the loop learned ──────────────────────────────────
# Extracts key concepts and shows how the question space evolved.
# Uses the same free models — no cost.

out = Path(OUTPUT_FILE)
if not out.exists():
    print('No data yet.')
elif not FREE_MODELS:
    print('No working models — run Cell 3 first.')
else:
    all_records = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]
    if len(all_records) < 5:
        print(f'Need at least 5 records — have {len(all_records)}.')
    else:
        # Ask the LLM to synthesize what was learned
        qa_summary = '\n'.join(
            f'Q{r["iteration"]}: {r["question"][:100]}'
            for r in all_records
        )
        synthesis_prompt = (
            f'Here are the questions generated by a self-improving learning loop '
            f'exploring "{TOPIC}":\n\n{qa_summary}\n\n'
            f'In 3-5 sentences: What did this loop discover? '
            f'What concepts did it converge on? '
            f'What is still unexplored?'
        )
        print('Synthesizing what the loop learned...')
        synthesis = ask(synthesis_prompt, FREE_MODELS[0], max_tokens=300)
        if synthesis:
            print()
            print('━' * 60)
            print('WHAT THE LOOP DISCOVERED')
            print('━' * 60)
            print(synthesis)
            print('━' * 60)
        else:
            print('Synthesis failed — try again.')

---

## What you built

A **self-referencing feedback loop**: each iteration's output shapes the next iteration's input.  
Time is the only parameter — the longer it runs, the more it knows.

## The genesis contract

Before running your own loop on a new topic, answer these four questions:

| Question | This notebook's answer |
|---|---|
| What are you reducing? | Gaps in understanding of the topic |
| What makes an iteration valid? | Answer introduces ≥1 new concept |
| When do you stop? | When you can explain the topic to a non-expert |
| What must never be sacrificed? | Answers must be checkable — no unfounded claims |

See: [The genesis contract, formally](https://pocoo.vaked.dev/posts/2026-06-24-genesis-contract-formally.html)

## Further reading

- [Your first free infinite loop](https://pocoo.vaked.dev/posts/2026-06-24-your-first-free-infinite-loop.html) — full blog post
- [The loop is already here](https://pocoo.vaked.dev/posts/2026-06-24-the-loop-is-already-here.html) — why this is new  
- [Reduce till it's a loop](https://pocoo.vaked.dev/posts/2026-06-24-reduce-till-its-a-loop.html) — the connection to compilers
- [Compressing the loop](https://pocoo.vaked.dev/posts/2026-06-24-compressing-the-loop.html) — making loops cheaper
- [Slop is data](https://pocoo.vaked.dev/posts/2026-06-24-slop-is-data.html) — on output quality
- [OpenRouter free models](https://openrouter.ai/models?max_price=0)
- [Unsloth fine-tuning](https://github.com/unslothai/unsloth)
- [Ollama local inference](https://ollama.ai)